# Macrocosm photo-z CNN — Kaggle runner

Trains the redshift CNN on the **sample_v3** 24×24×5 ugriz cutouts. Model/training logic lives in
`photoz_cnn.py` (cloned below) — edit it to change the architecture.

## Before running — notebook settings (right panel)
1. **Accelerator → GPU** (T4 ×2 or P100).
2. **Internet → On** (needed for `git clone`, `pip install mlflow`, and MLflow logging).
3. **Add Input → Datasets →** search **`macrocosm-sdss-ugriz-24px-v3`** and add it.
   (It mounts read-only at `/kaggle/input/macrocosm-sdss-ugriz-24px-v3/` with `images_*.npy` + `catalog_v1.parquet`.)


## 1. Setup + data
Set the cutout size **before** importing (eval/photoz_cnn read it at import), clone the code,
install MLflow, and locate the mounted dataset.


In [ ]:
import os, glob
os.environ['CUTOUT_SIZE'] = '24'        # sample_v3 is 24x24 (registered+cropped)

# model code + splits (needs Internet ON)
![ -d /kaggle/working/CNN-Model ] || git clone -q -b task-v3 https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /kaggle/working/CNN-Model
%cd /kaggle/working/CNN-Model
!pip -q install mlflow

# data = the added Kaggle dataset (images_*.npy + catalog_v1.parquet)
hits = glob.glob('/kaggle/input/*/catalog_v1.parquet')
assert hits, "Add the 'macrocosm-sdss-ugriz-24px-v3' dataset via Add Input first."
DATA_DIR = os.path.dirname(hits[0])
print('DATA_DIR =', DATA_DIR, '| shards =', len(glob.glob(f'{DATA_DIR}/images_*.npy')))

import tensorflow as tf
print('GPUs     =', tf.config.list_physical_devices('GPU'))

## 2. Train
`train()` loads the cutouts into RAM (~550k @ 24px ≈ 3.2 GB — fits), trains, and scores the fixed
50k val set. Paste the **MLflow token** (ask the team) to log params/metrics/outliers; `None` = train only.

Edit `photoz_cnn.py` (the `build_cnn` model cell) to try your own architecture, then re-run.


In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir=DATA_DIR,
    crop=24,               # sample_v3 is already 24x24
    preproc='p99',         # per-band / p99 — best so far (zscore val sigma_MAD ~0.025 -> p99 ~0.016)
    N=None,                # cap #train images if RAM-limited; None = all (~550k)
    batch=256, lr=3e-4, epochs=50,
    run_name='kaggle-run',
    experiment='oa',
    mlflow_token=None,     # paste the MLflow bearer token to log; None = train only
)
print(metrics)

## 3. (optional) 3-fold OOF outliers
Trains 3 models to find the hard/outlier training objects. **Note:** `cv_outliers.run` writes results
to `gs://…` via gsutil, which isn't authed on Kaggle — either skip this, or pass a local `--out` if you
patch it. Left commented.


In [ ]:
# from cv_outliers import run
# df = run(seed=0, data_dir=DATA_DIR, crop=24, preproc='p99')   # writes to GCS -> needs auth on Kaggle